# Терең оқыту негізінде банктік өтініштерді автоматты жіктеу жүйесін әзірлеу
«Терең оқыту» пәні бойынша курстық жұмыс.

Ағылшын тіліндегі өтінішті екі модель жіктейді. Алдымен олардың он санатты қаншалықты дәл ажырататынын салыстырамыз. Содан кейін модель сенімсіз жауап бергенде өтінішті операторға жолдау қандай нәтиже беретінін тексереміз.

Google Colab ортасында барлық ұяшықты ретімен іске қосыңыз. GPU қолжетімді болса, есептеу жылдамдайды. CPU да қолданылады. Алғашқы іске қосқанда кітапханалар, деректер мен модель салмақтары интернет арқылы жүктеледі.

## 1. Бағдарламалық ортаны дайындау
Мәтінді өңдеуге қажетті кітапханаларды орнатамыз. Colab ортасындағы үйлесімді PyTorch нұсқасы пайдаланылады. Дыбыс пен кескін өңдеуге арналған артық пакеттер нұсқалар арасындағы қайшылықты болдырмау үшін осы есептеу ортасынан жойылады.

In [ ]:
import os, sys, subprocess
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
                       'torchvision', 'torchaudio', 'librosa'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.6,<3', 'transformers==4.57.1', 'tokenizers==0.22.1',
    'datasets==4.4.1', 'accelerate==1.11.0', 'huggingface-hub==0.36.0',
    'gradio==5.49.1', 'scikit-learn==1.6.1',
    'matplotlib==3.10.0', 'seaborn==0.13.2', 'pandas>=2.2,<3'])
print('Кітапханалар орнатылды.')


In [ ]:
import json, random, shutil
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from datasets import load_dataset, DatasetDict, ClassLabel
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
import transformers, datasets, gradio as gr

SEED = 42
set_seed(SEED)
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('/content/banking77_project') if Path('/content').exists() else Path.cwd() / 'banking77_project'
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = ROOT / 'trained_model'
print('Есептеу құрылғысы:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__, '| Gradio:', gr.__version__)
sns.set_theme(style='whitegrid')

KK_LABELS = {'activate_my_card': 'Картаны белсендіру', 'lost_or_stolen_card': 'Картаның жоғалуы немесе ұрлануы', 'card_swallowed': 'Картаның банкоматта қалып қоюы', 'change_pin': 'PIN-кодты өзгерту', 'transaction_charged_twice': 'Бір сатып алу үшін екі рет ақша алынуы', 'declined_card_payment': 'Картамен төлемнің қабылданбауы', 'cancel_transfer': 'Ақша аударымын тоқтату', 'transfer_not_received_by_recipient': 'Аударымның алушыға жетпеуі', 'wrong_amount_of_cash_received': 'Банкоматтың қате сома беруі', 'card_arrival': 'Картаның жеткізілуін күту'}


## 2. BANKING77 деректерін жүктеу және зерттеу
BANKING77 жинағында банк клиенттерінің ағылшын тіліндегі өтініштері 77 санатқа бөлінген. Санаттардың атауларын categories.json файлынан оқимыз. Оларды әліпби ретімен орналастырып, бастапқы сандық белгілерін анықтаймыз.

[Деректер сипаттамасы](https://huggingface.co/datasets/PolyAI/banking77) мен [PolyAI жариялаған CSV файлдары](https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data) бір жинаққа жатады. CSV файлдарын Hugging Face Datasets арқылы оқып, авторлар ұсынған оқыту және тест жиындарын сақтаймыз. Дереккөз: Casanueva және т.б., 2020. Лицензиясы: CC BY 4.0.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/'
with urlopen(BASE_URL + 'categories.json', timeout=60) as response:
    source_categories = json.load(response)
all_names = sorted(source_categories, key=str.lower)
assert len(all_names) == len(set(all_names)) == 77
print('Бастапқы санат атаулары:')
print(all_names)
raw = load_dataset('csv', data_files={
    'train': BASE_URL + 'train.csv', 'test': BASE_URL + 'test.csv'})
assert set(raw['train'].column_names) == {'text', 'category'}
raw = raw.rename_column('category', 'label')
original_label = ClassLabel(names=all_names)
raw = raw.cast_column('label', original_label)
print('Өтініштердің жалпы саны:', sum(len(split) for split in raw.values()))
print('Санат саны:', len(all_names))
print({name: len(split) for name, split in raw.items()})
examples = raw['train'].select(range(5)).to_pandas()
examples['category'] = examples['label'].map(original_label.int2str)
display(examples)


In [ ]:
# Санат атауларын жүктелген деректер сұлбасымен салыстырамыз.
selected_names = list(KK_LABELS)
assert set(selected_names).issubset(raw['train'].features['label'].names)
old_to_new = {original_label.str2int(name): i for i, name in enumerate(selected_names)}
display(pd.DataFrame([{'Бастапқы нөмір': old, 'Жаңа нөмір': new,
    'Санат белгісі': selected_names[new], 'Қазақша атауы': KK_LABELS[selected_names[new]]}
    for old, new in old_to_new.items()]))
selected_raw = raw.filter(lambda row: row['label'] in old_to_new)
counts = pd.DataFrame({split: pd.Series(data['label']).value_counts()
                      for split, data in selected_raw.items()}).fillna(0).astype(int)
counts.index = [KK_LABELS[original_label.int2str(int(i))] for i in counts.index]
counts = counts.rename(columns={'train':'Оқыту','test':'Тест'})
display(counts)
print('Таңдалған санаттардағы өтініштер саны:', sum(len(x) for x in selected_raw.values()))
counts.plot.barh(figsize=(10, 6))
plt.title('BANKING77 жинағындағы он санаттың үлестірімі')
plt.xlabel('Өтініш саны')
plt.tight_layout()
plt.show()


## 3. Деректерді дайындау
Таңдалған он санатқа 0-ден 9-ға дейін жаңа нөмір береміз. Бастапқы оқыту жиынының 20 пайызын тексеруге бөлеміз. Әр санаттың үлесі сақталады. Тексеру жиыны модель мен шешім шегін таңдауға, ал тест жиыны соңғы бағалауға арналған.

Токенизатор сөздерді бөліктерге жіктеп, оларды сандық белгілерге айналдырады. Бір мәтіннің ұзындығы 128 токенмен шектеледі. Бір топтағы қысқа тізбектер сол топтың ең ұзын тізбегіне дейін толықтырылады.

In [ ]:
# Он санат үшін жаңа белгілер сұлбасын құрамыз.
selected = DatasetDict({split: data.map(
    lambda row: {'text': row['text'], 'labels': old_to_new[row['label']]},
    remove_columns=data.column_names)
    for split, data in selected_raw.items()})
selected = selected.cast_column('labels', ClassLabel(names=selected_names))
partition = selected['train'].train_test_split(
    test_size=0.2, seed=SEED, stratify_by_column='labels')
data = DatasetDict(train=partition['train'], validation=partition['test'], test=selected['test'])
assert set(data['train']['labels']) == set(range(10))
print({name: len(split) for name, split in data.items()})
display(pd.DataFrame({name: np.bincount(split['labels'], minlength=10)
                      for name, split in data.items()}, index=selected_names))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
encoded = data.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Бірінші мәтіннің токендері:', tokenizer.convert_ids_to_tokens(encoded['train'][0]['input_ids']))


### Салыстыруға арналған базалық әдіс
TF–IDF сөздер мен сөз тіркестерінің жиілігін сипаттайды. Логистикалық регрессия осы белгілер бойынша санатты болжайды. Екі нұсқаның бірін тексеру жиынындағы Macro F1 бойынша таңдаймыз.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from time import perf_counter

baseline = None
best_score = -1
baseline_runs = []
for max_ngram in (1, 2):
    candidate = make_pipeline(
        TfidfVectorizer(ngram_range=(1, max_ngram), sublinear_tf=True),
        LogisticRegression(max_iter=1000, random_state=SEED))
    start = perf_counter()
    candidate.fit(list(data['train']['text']), list(data['train']['labels']))
    elapsed = perf_counter() - start
    predictions = candidate.predict(list(data['validation']['text']))
    score = f1_score(data['validation']['labels'], predictions, average='macro')
    baseline_runs.append(dict(max_ngram=max_ngram, validation_macro_f1=score, fit_seconds=elapsed))
    if score > best_score:
        baseline, best_score = candidate, score
display(pd.DataFrame(baseline_runs))


## 4. DistilBERT моделін қосымша оқыту
Кодтаушы мәтіндегі сөздердің өзара байланысын ескеріп, сандық көрсетілім құрады. Жіктеу қабаты одан он санаттың ұпайларын есептейді. Бұл қабаттың бастапқы салмақтары кездейсоқ беріледі. AdamW оңтайландырғышы жіктеу қабаты мен кодтаушының салмақтарын бірге жаңартады.

Оқыту үш кезеңге созылады. Бастапқы оқу жылдамдығы 2 × 10⁻⁵, кездейсоқ сандар генераторының бастапқы мәні 42. Бір қадамда GPU арқылы 16 мәтін, CPU арқылы 8 мәтін өңделеді. Соңғы бағалау үшін тексеру жиынында Macro F1 көрсеткіші ең жоғары болған кезең таңдалады.

In [ ]:
def compute_metrics(prediction):
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicted, labels=list(range(10)), average='macro', zero_division=0)
    return {'accuracy': accuracy_score(labels, predicted),
            'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1}

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=10,
    id2label=dict(enumerate(selected_names)),
    label2id={name: i for i, name in enumerate(selected_names)},
    use_safetensors=True)
assert all(p.requires_grad for p in model.parameters())
print('Оқытылатын параметрлер саны:', sum(p.numel() for p in model.parameters() if p.requires_grad))
initial_classifier = model.classifier.weight.detach().cpu().clone()
args = TrainingArguments(
    output_dir=str(ROOT / 'checkpoints'), overwrite_output_dir=True,
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=16 if DEVICE.type == 'cuda' else 8,
    per_device_eval_batch_size=32 if DEVICE.type == 'cuda' else 8,
    weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy='epoch', save_strategy='epoch',
    logging_strategy='steps', logging_steps=10,
    load_best_model_at_end=True, metric_for_best_model='f1_macro',
    greater_is_better=True, save_total_limit=1,
    seed=SEED, data_seed=SEED, report_to='none',
    fp16=False, bf16=False, optim='adamw_torch',
    dataloader_num_workers=0, dataloader_pin_memory=DEVICE.type == 'cuda',
    use_cpu=DEVICE.type == 'cpu')
trainer = Trainer(model=model, args=args,
    train_dataset=encoded['train'], eval_dataset=encoded['validation'],
    processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
train_result = trainer.train()
assert trainer.state.global_step > 0
assert not torch.equal(initial_classifier, trainer.model.classifier.weight.detach().cpu())
print('Салмақтар жаңартылды. Орындалған қадам саны:', trainer.state.global_step)
print('Тексеру жиынындағы ең жоғары Macro F1:', trainer.state.best_metric)


### Операторға жіберу шегін таңдау
Тексеру жиынында автоматты қабылданған жауаптардың кемінде 98%-ы дұрыс болатын шектерді қарастырамыз. Солардың ішінен ең көп өтінішті қамтитын шекті таңдаймыз. Кемінде 30 қабылданған мысал қажет. Шарт орындалмаса, барлық өтініш операторға беріледі. 98% – тәжірибелік мақсат.

In [ ]:
from scipy.special import softmax

def routing_stats(scores, correct, cutoff):
    accepted = scores >= cutoff
    count = int(accepted.sum())
    return dict(threshold=float(cutoff), accepted=count, coverage=count / len(scores),
                accuracy=float(correct[accepted].mean()) if count else np.nan,
                errors=int((~correct[accepted]).sum()))

validation = trainer.predict(encoded['validation'])
val_scores = softmax(validation.predictions, axis=1).max(axis=1)
val_correct = validation.predictions.argmax(axis=1) == validation.label_ids
curve = pd.DataFrame([routing_stats(val_scores, val_correct, t)
                      for t in np.unique(np.append(val_scores, 0.0))])
eligible = curve.query('accuracy >= 0.98 and accepted >= 30')
threshold = float(eligible.iloc[0].threshold) if len(eligible) else 1.01
display(pd.DataFrame([routing_stats(val_scores, val_correct, threshold)]))
MODEL_DIR.mkdir(exist_ok=True, parents=True)
curve.to_csv(MODEL_DIR / 'validation_thresholds.csv', index=False)
curve.plot(x='coverage', y='accuracy', xlabel='Автоматты өңделген өтініштер үлесі',
           ylabel='Қабылданған жауаптардың дәлдігі', legend=False)
plt.axhline(0.98, linestyle='--', color='gray')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'routing_validation.png', dpi=150)
plt.show()

## 5. Тәуелсіз тест жиынында бағалау
Accuracy барлық дұрыс жауаптың үлесін көрсетеді. Precision модель белгілі бір санатқа жатқызған өтініштердің қаншасы дұрыс екенін, ал Recall сол санаттағы өтініштердің қаншасы табылғанын көрсетеді. F1 осы екі көрсеткішті біріктіреді. Macro орташа мәнін есептегенде барлық санатқа бірдей салмақ беріледі.

Тест нәтижесін көргеннен кейін осы жиын бойынша модель параметрлері мен шешім шегін өзгертпейміз.

In [ ]:
test_output = trainer.predict(encoded['test'], metric_key_prefix='test')
y_true = np.asarray(test_output.label_ids)
y_pred = test_output.predictions.argmax(axis=-1)
metrics = compute_metrics((test_output.predictions, y_true))
display(pd.DataFrame({'Көрсеткіш': list(metrics), 'Мәні': list(metrics.values())}))
kk_names = [KK_LABELS[name] for name in selected_names]
print(classification_report(y_true, y_pred, labels=list(range(10)),
                            target_names=kk_names, digits=4, zero_division=0))
report = classification_report(y_true, y_pred, labels=list(range(10)),
    target_names=selected_names, output_dict=True, zero_division=0)
(ROOT / 'test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(ROOT / 'classification_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(confusion_matrix(y_true, y_pred, labels=list(range(10))),
    annot=True, fmt='d', cmap='Blues', xticklabels=kk_names, yticklabels=kk_names, ax=ax)
ax.set(xlabel='Болжанған санат', ylabel='Нақты санат', title='Тест жиынындағы қателер матрицасы')
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.savefig(ROOT / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### Тәуелсіз тест жиынындағы салыстыру
Екі модель де бірдей өтініштерді жіктейді. Төмендегі мысалдар олардың қандай жағдайларда әртүрлі шешім қабылдағанын көрсетеді.

In [ ]:
baseline_pred = baseline.predict(list(data['test']['text']))
rows = []
for name, pred in [('TF–IDF + Logistic Regression', baseline_pred), ('DistilBERT', y_pred)]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average='macro', zero_division=0)
    rows.append(dict(model=name, accuracy=accuracy_score(y_true, pred),
                     macro_precision=precision, macro_recall=recall, macro_f1=f1))
comparison = pd.DataFrame(rows)
display(comparison.rename(columns={'model':'Модель','accuracy':'Дәлдік'}).round(4))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(MODEL_DIR / 'model_comparison.csv', index=False)
pd.DataFrame(baseline_runs).to_csv(MODEL_DIR / 'baseline_validation.csv', index=False)
examples = pd.DataFrame({'Мәтін': list(data['test']['text']),
                         'Нақты санат': [selected_names[i] for i in y_true],
                         'TF–IDF': [selected_names[i] for i in baseline_pred],
                         'DistilBERT': [selected_names[i] for i in y_pred]})
display(examples[baseline_pred != y_pred].head(10))


### Таныс және белгісіз санаттарды тексеру
Алдымен бекітілген шекті он таныс санаттың тест жиынында бағалаймыз. Содан кейін BANKING77 жиынындағы қалған 67 санатты бөлек сынаймыз. Бұл өтініштер үшін кез келген автоматты санат қате болады. Екі топтың жиілігі нақты банктегі сұраныс ағынын білдірмейді.

In [ ]:
known_scores = softmax(test_output.predictions, axis=1).max(axis=1)
unknown = raw['test'].filter(lambda row: row['label'] not in old_to_new)
unknown_encoded = unknown.remove_columns('label').map(
    tokenize, batched=True, remove_columns=['text'])
unknown_logits = trainer.predict(unknown_encoded).predictions
unknown_scores = softmax(unknown_logits, axis=1).max(axis=1)

results = []
for mode, cutoff in [('Барлығын жіктеу', 0.0), ('Операторға жолдау', threshold)]:
    for group, scores, correct in [
        ('Таныс санаттар', known_scores, y_pred == y_true),
        ('Белгісіз санаттар', unknown_scores, np.zeros(len(unknown), dtype=bool))]:
        row = routing_stats(scores, correct, cutoff)
        results.append(dict(mode=mode, group=group, total=len(scores),
                            to_operator=len(scores)-row['accepted'], **row))
routing_report = pd.DataFrame(results)
display(routing_report.rename(columns={'mode':'Режим','group':'Санаттар тобы','total':'Барлығы','to_operator':'Операторға','threshold':'Шек','accepted':'Автоматты','coverage':'Қамту үлесі','accuracy':'Дәлдік','errors':'Қате'}).round(4))
routing_report.to_csv(MODEL_DIR / 'routing_test.csv', index=False)
audit = pd.DataFrame({'text':list(unknown['text']),
                      'actual':[original_label.int2str(i) for i in unknown['label']],
                      'suggested':[selected_names[i] for i in unknown_logits.argmax(axis=1)],
                      'score':unknown_scores})
false_routes = audit[audit.score >= threshold].sort_values('score', ascending=False)
display(false_routes.head(5))
false_routes.to_csv(MODEL_DIR / 'unknown_false_routes.csv', index=False)

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
training_logs = history.dropna(subset=['loss'])
axes[0].plot(training_logs['step'], training_logs['loss'], marker='.')
axes[0].set(title='Оқыту шығыны (аралық бойынша орташа)', xlabel='Қадам', ylabel='Cross-entropy')
validation_logs = history.dropna(subset=['eval_loss'])
axes[1].plot(validation_logs['epoch'], validation_logs['eval_loss'], marker='o', color='darkorange')
axes[1].set(title='Тексеру шығыны', xlabel='Оқыту кезеңі', ylabel='Cross-entropy')
plt.tight_layout()
plt.savefig(ROOT / 'loss_curves.png', dpi=150)
plt.show()
history.to_json(ROOT / 'training_history.json', orient='records', indent=2)

# Әр санаттан нәтижеге тәуелсіз бір тест мысалы алынады.
rng = np.random.default_rng(SEED)
indices = [int(rng.choice(np.flatnonzero(y_true == i))) for i in range(10)]
display(pd.DataFrame([{'Өтініш': data['test'][i]['text'],
    'Нақты санат': kk_names[y_true[i]],
    'Болжам': kk_names[y_pred[i]], 'Дұрыс': bool(y_true[i] == y_pred[i])}
    for i in indices]))


## 6. Модельді сақтау және қайта жүктеу
Архивке модель салмақтары, токенизатор, қазақша санат атаулары, шешім шегі және тәжірибе нәтижелері жазылады. Интерфейс дискіден жүктелген модельді қолданады.

Басқа сеанста оқытусыз пайдалану үшін алдымен 1-бөлімді орындаңыз. Сақталған ZIP файлын Colab ортасына жүктеп, `shutil.unpack_archive('/content/banking77_distilbert.zip', str(MODEL_DIR))` пәрменімен ашыңыз. Содан кейін төмендегі сақталған модельді жүктеу ұяшығын және 7-бөлімді іске қосыңыз.

In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
metadata = {'base_model': MODEL_NAME, 'seed': SEED, 'max_length': MAX_LENGTH,
            'kk_labels': KK_LABELS, 'test_metrics': metrics,
            'dataset': 'PolyAI/banking77', 'epochs': 3, 'routing_threshold': threshold,
            'split_sizes': {k: len(v) for k, v in data.items()}}
(MODEL_DIR / 'project_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
for filename in ['test_metrics.json', 'classification_report.json',
                 'training_history.json', 'loss_curves.png', 'confusion_matrix.png']:
    shutil.copy2(ROOT / filename, MODEL_DIR / filename)
archive_path = shutil.make_archive(str(ROOT / 'banking77_distilbert'), 'zip', MODEL_DIR)
print('Модель сақталған орын:', MODEL_DIR)
print('Архив:', archive_path)


In [ ]:
# Сақталған модельді жүктейміз.
saved_metadata = json.loads((MODEL_DIR / 'project_metadata.json').read_text(encoding='utf-8'))
ui_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
ui_model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR), local_files_only=True).to(DEVICE)
ui_model.eval()
ui_labels = saved_metadata.get('kk_labels', KK_LABELS)
archive_path = str(ROOT / 'banking77_distilbert.zip')
if not Path(archive_path).exists():
    archive_path = shutil.make_archive(str(ROOT / 'banking77_distilbert'), 'zip', MODEL_DIR)
print('Сақталған модель жүктелді.')


## 7. Өтінішті тексеру
Ағылшын тілінде өтініш жазыңыз немесе мысалдардың бірін таңдаңыз. Жүйе ең ықтимал үш санатты және модельдің сенімділік бағасын көрсетеді. Баға шектен төмен болса, өтінішті операторға жіберуді ұсынады.

Gradio интерфейсі ұяшықтың астында ашылады. Colab сеансы жұмыс істеп тұрғанда уақытша сілтеме арқылы да ашуға болады. Мысалға нақты клиенттің жеке мәліметтерін енгізбеңіз.

In [ ]:
def classify_request(text):
    if not text or not text.strip():
        return 'Өтінішіңізді ағылшын тілінде жазыңыз.', '', {}
    inputs = ui_tokenizer(text.strip(), return_tensors='pt', truncation=True,
                          max_length=saved_metadata['max_length'])
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        probabilities = torch.softmax(ui_model(**inputs).logits[0], dim=-1).cpu()
    values, indices = probabilities.topk(3)
    top3 = {name: float(p)
            for p, i in zip(values, indices)
            for name in [ui_labels[ui_model.config.id2label[int(i)]]]}
    winner = next(iter(top3))
    decision = winner if float(values[0]) >= saved_metadata["routing_threshold"] else "Операторға жіберу"
    return decision, f'{float(values[0]) * 100:.2f} %', top3

with gr.Blocks(title='Банктік өтініштерді автоматты жіктеу', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Банктік өтініштерді автоматты жіктеу')
    gr.Markdown('Өтінішіңізді ағылшын тілінде жазыңыз. Жүйе BANKING77 жиынындағы он санатты ажыратады.')
    input_text = gr.Textbox(label='Өтініш мәтіні', lines=3,
                           placeholder='I lost my bank card yesterday.')
    button = gr.Button('Санатты анықтау', variant='primary')
    category = gr.Textbox(label='Жүйенің шешімі', interactive=False)
    confidence = gr.Textbox(label='Модельдің сенімділік бағасы', interactive=False)
    top_categories = gr.Label(label='Ең ықтимал үш санат', num_top_classes=3)
    button.click(classify_request, inputs=input_text, outputs=[category, confidence, top_categories])
    input_text.submit(classify_request, inputs=input_text, outputs=[category, confidence, top_categories])
    gr.Examples(examples=[['I lost my bank card yesterday.'],
        ['My card was charged twice for the same purchase.'],
        ['What exchange rate will I get?'], ['How do I activate my new card?'], ['The ATM kept my card.'],
        ['I want to cancel a bank transfer.'], ['How can I change my PIN?']],
        inputs=input_text, label='Мысалды таңдап, «Санатты анықтау» батырмасын басыңыз')
    gr.Markdown('Сенімділік бағасы төмен болса, жүйе өтінішті операторға жіберуді ұсынады. Жоғары баға берілгенде де модель қателесуі мүмкін.')
    gr.File(value=archive_path, label='Оқытылған модельді, токенизаторды және нәтижелерді жүктеп алу', interactive=False)
demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, prevent_thread_lock=True)


## 8. Қорытынды
Он таныс санат үшін TF–IDF пен логистикалық регрессияны DistilBERT моделімен салыстырдық. Операторға жолдау әсерін бағалау үшін дұрыс жауап үлесімен қатар автоматты өңделген өтініштер санын да есептедік.

Оқытуға кірмеген 67 санат бөлек тексеріледі. Мұндай өтінішті модель сенімді түрде қате жіктеуі мүмкін. Нақты банктегі автоматты өңдеу үлесі клиенттер қандай тақырыптарда жазатынына байланысты. Келесі ұяшық сақталған модельдің осы іске қосылымдағы нәтижелерін көрсетеді.

In [ ]:
measured = saved_metadata['test_metrics']
display(Markdown(
    '**Сақталған модельдің тест нәтижелері:**  \n'
    + '  \n'.join(f'- {name}: **{value:.4f}**' for name, value in measured.items())))


### Дереккөздер
- [BANKING77 жинағының сипаттамасы](https://huggingface.co/datasets/PolyAI/banking77)
- [PolyAI жариялаған бастапқы деректер](https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data)
- [DistilBERT моделі](https://huggingface.co/distilbert/distilbert-base-uncased)
- [Trainer және TrainingArguments құжаттамасы](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)

Орындалған ұяшықтардың нәтижелерін сақтау үшін тәжірибе аяқталған соң ноутбукты IPYNB пішімінде жүктеп алыңыз.